# Smart MCQ Multiple Choice (Q1-Q10)




In [13]:
# Install compatible libraries
!pip -q uninstall -y torchao
!pip -q install -U transformers datasets accelerate peft==0.15.2 sentencepiece

import os
os.environ["TOKENIZERS_PARALLELISM"]="false"

import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments,
    DataCollatorForMultipleChoice,
)

from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

train=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

mapping={"A":0,"B":1,"C":2,"D":3,"E":4}
train["label"]=train["answer"].map(mapping)
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")


## Q1

In [14]:
print(train.loc[150,'label'])

2


## Q2

In [15]:
row=train.loc[0]
txt=str(row.prompt)+" [SEP] "+str(row.B)
print(len(txt))


407


## Q3

In [16]:
row=train.loc[0]
choices=[f"{row.prompt} [SEP] {row[c]}" for c in ["A","B","C","D","E"]]
enc=tokenizer(choices,padding="max_length",truncation=True,max_length=128,return_tensors="pt")
input_ids=enc["input_ids"].unsqueeze(0)
attn=enc["attention_mask"].unsqueeze(0)
print(input_ids.shape)
print(input_ids.shape[1])


torch.Size([1, 5, 128])
5


## Q4

In [17]:
ids=[]
for _,r in train.head(16).iterrows():
    ch=[f"{r.prompt} [SEP] {r[c]}" for c in ["A","B","C","D","E"]]
    ids.append(tokenizer(ch,padding="max_length",truncation=True,max_length=128,return_tensors="pt")["input_ids"])
ids=torch.stack(ids)
print(ids.shape)
print(ids.numel())


torch.Size([16, 5, 128])
10240


## Q5 & Q6

In [18]:
model=AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
out=model(input_ids=input_ids,attention_mask=attn)
print("Logits shape:",out.logits.shape)
print("Logits/question:",out.logits.shape[1])

label=torch.tensor([train.loc[0,"label"]])
out=model(input_ids=input_ids,attention_mask=attn,labels=label)
print("Loss shape:",out.loss.shape)
print("Loss ndim:",out.loss.ndim)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: torch.Size([1, 5])
Logits/question: 5
Loss shape: torch.Size([])
Loss ndim: 0


## Q7

In [19]:
config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model=get_peft_model(model,config)
trainable=sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(trainable)


295681


## Q8

In [20]:
def preprocess(ex):
    texts=[f"{ex['prompt']} [SEP] {ex[c]}" for c in ["A","B","C","D","E"]]
    tok=tokenizer(texts,padding="max_length",truncation=True,max_length=128)
    return {
        "input_ids":tok["input_ids"],
        "attention_mask":tok["attention_mask"],
        "labels":mapping[ex["answer"]]
    }

ds=Dataset.from_pandas(train.head(100))
ds=ds.map(preprocess)
print(torch.tensor(ds[0]["input_ids"]).shape)
print(len(ds[0]["input_ids"]))


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

torch.Size([5, 128])
5


## Q9 & Q10

In [22]:
print("Use the dataset above with DataCollatorForMultipleChoice.")
print("TrainingArguments(max_steps=4, per_device_train_batch_size=4)")
print("trainer.train()")
print("Answer Q9 with trainer.state.global_step")
print("Answer Q10 with:")
print()


Use the dataset above with DataCollatorForMultipleChoice.
TrainingArguments(max_steps=4, per_device_train_batch_size=4)
trainer.train()
Answer Q9 with trainer.state.global_step
Answer Q10 with:

